<a href="https://colab.research.google.com/github/harshkumar8a/fine-tuning-using-mlflow/blob/main/Fine_Tuning_with_Mlflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine Tuning with Mlflow

  *   Here, I loaded the dataset using **load_dataset()**
  *   Used the pre-trained model, which is **bert-base-uncased** get from the Hugging face
  *   Tokenizer **AutoTokenizer.from_pretrained(MODEL_NAME)**
  *   Also using the **TrainingArgument()**, **Trainer**
  *   Then Applying the mlflow for fine tuning, changes some value

## Install Dependices

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
import transformers
import numpy as np
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
import torch


Checking the transfromers version

In [ ]:
print(transformers.__version__)

5.0.0


Checking the device is **cuda** or **cpu**

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## Load the IMDB datasets

In [ ]:
# Load IMDb sentiment dataset (25k train, 25k test, labels: 0=neg, 1=pos)
dataset = load_dataset("imdb")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
# Load BERT tokenizer
MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",   # pad shorter sequences
        truncation=True,         # cut sequences > 512 tokens
        max_length=128           # BERT max is 512; 128 is faster for demos
    )

# Apply tokenization to all splits
tokenized = dataset.map(tokenize_fn, batched=True)

# Remove the raw text column — Trainer only needs token IDs
tokenized = tokenized.remove_columns(["text"])
tokenized = tokenized.rename_column("label", "labels")  # Trainer expects "labels"
tokenized.set_format("torch")  # Return PyTorch tensors

print(tokenized["train"][0].keys())
# dict_keys(['input_ids', 'attention_mask', 'token_type_ids', 'labels'])

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

dict_keys(['labels', 'input_ids', 'token_type_ids', 'attention_mask'])


## Load pre-trained BERT with a fresh 2-class classification head


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2  # negative / positive
)
model.to(device)
print(model.device)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


cuda:0


## Metric function called at the end of each eval epoch


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted")
    }

## TrainingArguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,            # small LR prevents catastrophic forgetting
    weight_decay=0.01,             # L2 regularization

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1",

    report_to="none"
)


## Use only 2000 samples for speed in this demo


In [ ]:
small_train = tokenized["train"].shuffle(seed=42).select(range(2000))
small_eval  = tokenized["test"].shuffle(seed=42).select(range(500))

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_eval,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.403419,0.832000,0.830701
2,No log,0.423883,0.832000,0.830450
3,No log,0.397847,0.860000,0.859827


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=375, training_loss=0.33953076171875, metrics={'train_runtime': 187.8765, 'train_samples_per_second': 31.936, 'train_steps_per_second': 1.996, 'total_flos': 394666583040000.0, 'train_loss': 0.33953076171875, 'epoch': 3.0})

# Using MLflow

In [ ]:
!pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 22.1 MB/s eta 0:00:00


In [2]:
%pip install -q dagshub mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8

In [3]:
import dagshub
dagshub.init(repo_owner='harshkumar8a', repo_name='fine-tuning-using-mlflow', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=4b570103-9f36-49dc-a49e-96d80b1731b6&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=2d91b28139408c211755fc4fa054e11b6d4dc4e38c41a6d5e016384063aaf094




Accessing as harshkumar8a

Initialized MLflow to track repo "harshkumar8a/fine-tuning-using-mlflow"

Repository harshkumar8a/fine-tuning-using-mlflow initialized!

In [4]:
import mlflow
import mlflow.pytorch
import numpy as np
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
import urllib3

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

🏃 View run worried-owl-789 at: https://dagshub.com/harshkumar8a/fine-tuning-using-mlflow.mlflow/#/experiments/0/runs/e8e39edd45a1464bb7380670af392626
🧪 View experiment at: https://dagshub.com/harshkumar8a/fine-tuning-using-mlflow.mlflow/#/experiments/0


In [ ]:
# Test logging
with mlflow.start_run():
    mlflow.log_param("test_param", "test_value")
    print("✓ Successfully connected to MLflow!")

✓ Successfully connected to MLflow!
🏃 View run useful-sloth-184 at: https://dagshub.com/harshkumar8a/fine-tuning-using-mlflow.mlflow/#/experiments/0/runs/063d58c17093493891d5d2db17f4c054
🧪 View experiment at: https://dagshub.com/harshkumar8a/fine-tuning-using-mlflow.mlflow/#/experiments/0


Configuration, which we can changes and see which value giving the best results

In [8]:
CONFIG = {
    "model_name":   "bert-base-uncased",
    "num_labels":   2,
    "max_length":   128,
    "batch_size":   128,
    "epochs":       8,
    "learning_rate": 2e-5,
    "weight_decay": 0.05,
    "train_size":   2000,
    "eval_size":    500,
}

In [7]:
dataset   = load_dataset("imdb")
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

...

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length",
                     truncation=True, max_length=CONFIG["max_length"])

tokenized = dataset.map(tokenize_fn, batched=True)
tokenized = tokenized.remove_columns(["text"])
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch")

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [10]:
train_ds = tokenized["train"].shuffle(seed=42).select(range(CONFIG["train_size"]))
eval_ds  = tokenized["test"].shuffle(seed=42).select(range(CONFIG["eval_size"]))

In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1":       f1_score(labels, preds, average="weighted"),
    }

In [12]:
mlflow.set_experiment("bert-sentiment-classification")

<Experiment: artifact_location='mlflow-artifacts:/fcdcc2fba4ec44b3bf8f27e2d63aa534', creation_time=1774501379419, experiment_id='1', last_update_time=1774501379419, lifecycle_stage='active', name='bert-sentiment-classification', tags={}, workspace='default'>

In [13]:
with mlflow.start_run(run_name="bert-base-5th-attend") as run:

    # Log every hyperparameter — you'll thank yourself when comparing runs
    mlflow.log_params(CONFIG)

    # Tag run with context metadata
    mlflow.set_tags({
        "dataset":     "imdb",
        "task":        "sentiment-classification",
        "framework":   "huggingface-transformers",
        "developer":   "Harsh Kumar",
    })

    # Build model
    model = AutoModelForSequenceClassification.from_pretrained(
        CONFIG["model_name"], num_labels=CONFIG["num_labels"]
    )
    model.to(device)



    training_args = TrainingArguments(
        output_dir       = f"./results/{run.info.run_id}",  # unique per run!
        num_train_epochs = CONFIG["epochs"],
        per_device_train_batch_size = CONFIG["batch_size"],
        per_device_eval_batch_size  = CONFIG["batch_size"] * 2,
        learning_rate    = CONFIG["learning_rate"],
        weight_decay     = CONFIG["weight_decay"],
        eval_strategy = "epoch",
        save_strategy       = "epoch",
        load_best_model_at_end = True,
        metric_for_best_model  = "f1",
        report_to = "none",   # disable default reporting; we log manually
    )

    from transformers import TrainerCallback

    class MLflowMetricsCallback(TrainerCallback):
        def on_evaluate(self, args, state, control, metrics=None, **kwargs):
            if metrics:
                epoch = int(state.epoch)
                mlflow.log_metric("eval_accuracy", metrics.get("eval_accuracy", 0), step=epoch)
                mlflow.log_metric("eval_f1",       metrics.get("eval_f1", 0),       step=epoch)
                mlflow.log_metric("eval_loss",     metrics.get("eval_loss", 0),     step=epoch)

        def on_log(self, args, state, control, logs=None, **kwargs):
            if logs and "loss" in logs:
                mlflow.log_metric("train_loss", logs["loss"], step=int(state.global_step))

    trainer = Trainer(
        model           = model,
        args            = training_args,
        train_dataset   = train_ds,
        eval_dataset    = eval_ds,
        compute_metrics = compute_metrics,
        callbacks       = [MLflowMetricsCallback()],
    )

    trainer.train()

    # 6. Final evaluation
    final_metrics = trainer.evaluate()
    mlflow.log_metrics({
        "final_accuracy": final_metrics["eval_accuracy"],
        "final_f1":       final_metrics["eval_f1"],
    })

    # 7. Save model as MLflow artifact
    # Option A: save as HuggingFace pipeline (easy to load later)
    from transformers import pipeline
    clf_pipeline = pipeline(
        "text-classification",
        model=trainer.model,
        tokenizer=tokenizer
    )
    mlflow.transformers.log_model(
        transformers_model=clf_pipeline,
        artifact_path="model",              # folder name inside the run
        task="text-classification",
    )

    print(f"Run ID: {run.info.run_id}")
    print(f"Final F1: {final_metrics['eval_f1']:.4f}")
    print(f"Final Accuracy: {final_metrics['eval_accuracy']:.4f}")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.603613,0.704000,0.698759
2,No log,0.476118,0.804000,0.802875
3,No log,0.394940,0.840000,0.839803
4,No log,0.401184,0.826000,0.825251
5,No log,0.373687,0.840000,0.839754
6,No log,0.364253,0.852000,0.851988
7,No log,0.367315,0.850000,0.849980
8,No log,0.378728,0.848000,0.847891


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

2026/03/27 04:39:10 WARNING mlflow.transformers: Could not infer model execution engine type due to huggingface_hub not being installed or unable to connect in online mode. Adding both Pytorchand Tensorflow to requirements.
Failure cause: cannot import name 'FlaxPreTrainedModel' from 'transformers' (/usr/local/lib/python3.12/dist-packages/transformers/__init__.py)
2026/03/27 04:39:10 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/03/27 04:39:10 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.25.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.25.0' without the local version label to mak

Run ID: eebe97c5b1274ec295114485f32f1349
Final F1: 0.8520
Final Accuracy: 0.8520
🏃 View run bert-base-5th-attend at: https://dagshub.com/harshkumar8a/fine-tuning-using-mlflow.mlflow/#/experiments/1/runs/eebe97c5b1274ec295114485f32f1349
🧪 View experiment at: https://dagshub.com/harshkumar8a/fine-tuning-using-mlflow.mlflow/#/experiments/1
